[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch02-ml-systems/04-do-tren-cpu-tinh-toan-hay-bo-nho.ipynb)

# Đo trên CPU: nghẽn ở tính toán hay nghẽn ở bộ nhớ

Chương nói cùng một mô hình có thể nghẽn ở tính toán khi huấn luyện mà nghẽn ở bộ nhớ khi suy luận (inference), và
nguyên lý điểm nghẽn bảo ta tìm số hạng chi phối trước khi tối ưu. Ba sổ tay trước tính điều đó cho phần
cứng của sách. Sổ tay này đo nó trên chính máy bạn đang chạy: một CPU cũng có hợp đồng với silicon của
nó, với một thông lượng đỉnh, một băng thông bộ nhớ (bandwidth) và một phụ trội cố định (fixed overhead).

Mọi con số đo ở đây là của máy này, mỗi máy một khác. Chúng không dùng để kiểm con số nào của sách.

**Bạn sẽ làm:**
1. đo thông lượng đỉnh và băng thông của máy, rồi suy ra điểm gãy (ridge point) của nó;
2. đo cả ba số hạng của định luật sắt trên một phép cộng hai mảng, từ vài KB tới vài trăm MB;
3. quét batch của một tầng tuyến tính, xem nó đi từ nghẽn ở bộ nhớ sang nghẽn ở tính toán;
4. đặt mọi phép toán vừa đo lên biểu đồ roofline của máy;
5. đo kiểu tải Sparse Scatter: tra một bảng embedding lớn theo ba thứ tự khác nhau;
6. dùng hai con số đo được để ước lượng thời gian sinh một token của GPT-2 trên máy này.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # thứ tự màu cố định, như các sổ tay khác
B4 = 4                                               # byte mỗi số FP32


def thoi_gian(f, lap=5):
    """Trung vị thời gian của f(), sau một lần chạy khởi động."""
    f()
    do = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        do.append(time.perf_counter() - t0)
    return float(np.median(do))


print(f"đo với {torch.get_num_threads()} luồng CPU")

## 1. Hai con số của máy

Thông lượng đỉnh: nhân hai ma trận 1024 × 1024, tốn $2N^3$ phép tính. Mỗi số được nạp về rồi dùng lại
hàng nghìn lần, nên phép này gần như chỉ đo tốc độ tính.

Băng thông: cộng hai mảng 32 triệu số FP32, mỗi mảng 128 MB. Mỗi phần tử đọc 8 byte, ghi 4 byte và chỉ
làm một phép cộng, nên phép này gần như chỉ đo tốc độ chở dữ liệu. Sổ tay đếm 12 byte mỗi phần tử, tức số
byte mà chương trình yêu cầu; phần cứng có thể chở nhiều hơn thế.

**Dự đoán:** tính theo phép tính mỗi giây, phép nào nhanh hơn, và hơn khoảng bao nhiêu lần?

In [ ]:
N_MM = 1024
A = torch.randn(N_MM, N_MM)
B_ = torch.randn(N_MM, N_MM)
C_ = torch.empty(N_MM, N_MM)
t_mm = thoi_gian(lambda: torch.mm(A, B_, out=C_), lap=5)
R_DINH = 2 * N_MM ** 3 / t_mm                  # phép tính mỗi giây

N_ADD = 1 << 25
a = torch.full((N_ADD,), 1.0)
b = torch.full((N_ADD,), 2.0)
c = torch.empty(N_ADD)
t_add = thoi_gian(lambda: torch.add(a, b, out=c), lap=5)
BW = 3 * B4 * N_ADD / t_add                    # byte mỗi giây
DIEM_GAY = R_DINH / BW                         # phép tính mỗi byte

do_tren_may("nhân ma trận 1024 × 1024", R_DINH / 1e9, "GFLOP/s")
do_tren_may("cộng hai mảng 128 MB, tính theo phép tính", N_ADD / t_add / 1e9, "GFLOP/s")
do_tren_may("cộng hai mảng 128 MB, tính theo byte", BW / 1e9, "GB/s")
do_tren_may("điểm gãy của máy, thông lượng đỉnh ÷ băng thông", DIEM_GAY, "FLOP mỗi byte")

Hai phép trên chạy cùng một bộ xử lý, nhưng một phép bị chặn bởi số học, phép kia bởi bộ nhớ. Tỉ số của
hai con số là điểm cân bằng máy (machine balance point) của chương: một phép toán có mật độ tính toán
(arithmetic intensity), tức số phép tính trên mỗi byte chở, thấp hơn điểm gãy thì nghẽn ở bộ nhớ (memory
bound), cao hơn thì nghẽn ở tính toán (compute bound).

## 2. Ba số hạng của định luật sắt, trên một phép cộng

Với một tác vụ đơn lẻ, chương dùng dạng cộng của định luật sắt:

$$T = \frac{D_\text{vol}}{\text{BW}} + \frac{O}{R_\text{peak}\cdot\eta_\text{hw}} + L_\text{lat}$$

Phép cộng hai mảng có số hạng tính toán rất nhỏ, nên còn lại hai số hạng: phụ trội cố định $L_\text{lat}$
của mỗi lần gọi (Python, PyTorch, cấp phát công việc), và thời gian chở $D_\text{vol}/\text{BW}$.

**Dự đoán:** mảng nhỏ đi 1,000 lần thì phép cộng có nhanh lên 1,000 lần không?

In [ ]:
co = [1 << k for k in range(8, 26)]          # 256 tới 32 triệu phần tử
t_co = []
for n in co:
    x, y, z = torch.full((n,), 1.0), torch.full((n,), 2.0), torch.empty(n)
    lap = 200 if n < 1 << 16 else (30 if n < 1 << 21 else 5)
    t_co.append(thoi_gian(lambda: torch.add(x, y, out=z), lap))
del x, y, z
byte_co = np.array([3 * B4 * n for n in co], dtype=float)
t_co = np.array(t_co)
L_LAT = t_co[:3].min()                        # phụ trội cố định: mảng nhỏ nhất gần như chỉ còn nó
bw_co = byte_co / t_co
do_tren_may("phụ trội cố định của một lần gọi torch.add", L_LAT * 1e6, "µs")
do_tren_may("băng thông cao nhất trong lần quét", bw_co.max() / 1e9, "GB/s")
do_tren_may("ở cỡ dữ liệu", byte_co[bw_co.argmax()] / 1e6, "MB")
do_tren_may("băng thông ở mảng lớn nhất", bw_co[-1] / 1e9, "GB/s")

In [ ]:
mo_hinh = L_LAT + byte_co / BW
fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.loglog(byte_co / 1e6, t_co * 1e6, "o-", color=MAU[0], lw=2, ms=6, mec="white", mew=1.2,
          label="đo trên máy này")
ax.loglog(byte_co / 1e6, mo_hinh * 1e6, color="0.35", lw=1.2, ls="--",
          label="định luật sắt: L + D / BW, với BW của mảng 128 MB")
ax.axhline(L_LAT * 1e6, color="0.6", lw=0.8, ls=":")
ax.text(1, L_LAT * 1e6 * 1.2, "phụ trội cố định của một lần gọi", fontsize=8, color="0.35")
ax.set_xlabel("dữ liệu mỗi lần gọi, D (MB, thang log)")
ax.set_ylabel("thời gian một lần gọi (µs, thang log)")
ax.set_title("torch.add từ vài KB tới vài trăm MB (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Bên trái, đường gần như nằm ngang: phụ trội cố định chi phối, nên mảng nhỏ hơn không nhanh hơn. Bên phải,
đường đi theo độ dốc 1: thời gian tỉ lệ với số byte, đúng số hạng $D_\text{vol}/\text{BW}$. Nếu ở giữa có
một đoạn nằm dưới đường gạch, nghĩa là ở đó máy chở nhanh hơn băng thông của mảng 128 MB: dữ liệu đủ nhỏ
để nằm trong cache. Đó là phân cấp bộ nhớ (memory hierarchy), và bảng độ trễ của chương ghi cùng ý ấy: một lần trúng
cache L1 khoảng 1 ns, một lần đọc DRAM lâu hơn nhiều.

Bài học cho tải nhỏ, kiểu Tiny Constraint: khi mỗi lần gọi chỉ chở vài KB, tối ưu băng thông không giúp
gì, vì phần lớn thời gian là phụ trội cố định.

## 3. Quét batch: từ Bandwidth Hog sang Compute Beast

Một tầng tuyến tính 4096 → 8192 có 33.6 triệu trọng số, 134 MB ở FP32. Với batch $B$, nó làm
$2 \cdot B \cdot 4096 \cdot 8192$ phép tính, nhưng trọng số chỉ phải chở một lần cho cả batch. Vì vậy mật
độ tính toán tăng gần như tỉ lệ với $B$:

$$I(B) = \frac{2BKM}{4\,(KM + BK + BM)} \approx \frac{B}{2} \quad\text{khi } B \ll K, M$$

Ở batch 1, đây là một phép nhân ma trận với vector, giống một bước sinh token: kiểu Bandwidth Hog. Chương
nói tăng batch dùng lại dữ liệu tốt hơn và đẩy một phép toán nghẽn ở bộ nhớ về phía nghẽn ở tính toán,
nhưng đồng thời tăng lượng dữ liệu làm việc và thời gian phục vụ cả batch.

Đường dự đoán dưới đây là dạng lấy giá trị lớn nhất của định luật sắt, dựng từ đúng hai con số đo ở
phần 1: $T(B) = \max\big(D(B)/\text{BW},\; O(B)/R_\text{peak}\big)$, coi $\eta_\text{hw} = 1$.

**Dự đoán:** từ batch 1 lên batch 8, thời gian một batch tăng gấp 8, hay gần như không đổi?

In [ ]:
del a, b, c
K_IN, M_OUT = 4096, 8192
WT = torch.full((K_IN, M_OUT), 0.01)          # trọng số, lưu liền theo (vào × ra)
CAC_B = [1, 2, 4, 8, 16, 32, 64, 128, 256]


def flop_tt(B):
    return 2 * B * K_IN * M_OUT


def byte_tt(B):
    """Giả định: trọng số, đầu vào và đầu ra mỗi thứ đi qua bộ nhớ đúng một lần."""
    return B4 * (K_IN * M_OUT + B * K_IN + B * M_OUT)


def mat_do(B):
    return flop_tt(B) / byte_tt(B)


assert abs(mat_do(1) - 0.5) < 0.001 and abs(mat_do(256) / 256 - 0.5) < 0.07   # I(B) ≈ B/2
t_b = []
for B in CAC_B:
    X = torch.full((B, K_IN), 1.0)
    t_b.append(thoi_gian(lambda: X @ WT, lap=5 if B <= 64 else 3))
t_b = np.array(t_b)
du_doan = np.array([max(byte_tt(B) / BW, flop_tt(B) / R_DINH) for B in CAC_B])
b_gay = next((B for B in range(1, 4097) if mat_do(B) >= DIEM_GAY), None)
for B, t in zip(CAC_B, t_b):
    do_tren_may(f"batch {B:>3}: một batch {t * 1e3:8.3f} ms, mỗi mẫu", t / B * 1e6, "µs")
do_tren_may("batch 8 so với batch 1, thời gian một batch", t_b[CAC_B.index(8)] / t_b[0], "lần")
print(f"theo hai con số đo được, tầng này chuyển sang nghẽn ở tính toán từ batch {b_gay} (đo trên máy này)"
      if b_gay else "theo hai con số đo được, tầng này nghẽn ở bộ nhớ ở mọi batch tới 4096 (đo trên máy này)")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 4))
a1.loglog(CAC_B, t_b * 1e3, "o-", color=MAU[0], lw=2, ms=6, mec="white", mew=1.2, label="đo trên máy này")
a1.loglog(CAC_B, du_doan * 1e3, color="0.35", lw=1.2, ls="--", label="định luật sắt, dạng lớn nhất")
a1.set_ylabel("thời gian một batch (ms)")
a1.set_title("Độ trễ cả batch tăng theo B")
a2.semilogx(CAC_B, t_b / CAC_B * 1e6, "o-", color=MAU[1], lw=2, ms=6, mec="white", mew=1.2, label="đo trên máy này")
a2.semilogx(CAC_B, du_doan / CAC_B * 1e6, color="0.35", lw=1.2, ls="--", label="định luật sắt, dạng lớn nhất")
a2.set_ylim(0, max(t_b / CAC_B) * 1e6 * 1.15)
a2.set_ylabel("thời gian mỗi mẫu (µs)")
a2.set_title("Thời gian mỗi mẫu giảm theo B")
for ax in (a1, a2):
    ax.set_xticks(CAC_B, [str(B) for B in CAC_B])
    ax.minorticks_off()
    ax.set_xlabel("batch B")
    if b_gay and b_gay <= CAC_B[-1]:
        ax.axvline(b_gay, color="0.7", lw=0.9, ls=":")
    ax.legend(loc="upper left" if ax is a1 else "upper right", fontsize=8)
if b_gay and b_gay <= CAC_B[-1]:
    a2.text(b_gay * 1.08, a2.get_ylim()[1] * 0.8, f"điểm gãy: batch {b_gay}", fontsize=8, color="0.4")
fig.suptitle("Tầng tuyến tính 4096 → 8192, FP32 (đo trên máy này)", fontsize=10)
plt.tight_layout()
plt.show()

Bên trái điểm gãy, đường dự đoán nằm ngang: thời gian là thời gian chở 134 MB trọng số, và thêm mẫu vào
batch gần như miễn phí. Bên phải, nó tăng theo $B$, vì giờ số học mới là điểm nghẽn. Khoảng cách giữa
đường đo và đường dự đoán là phần mà hai con số của phần 1 không nắm được: $\eta_\text{hw}$ của từng kích
thước, và việc một phép toán chở byte hiệu quả hơn hay kém hơn phép cộng. Nếu đường đo có chỗ gãy khúc ở
batch nhỏ, thường đó là lúc thư viện đổi sang một kernel khác theo kích thước, không phải lúc phần cứng đổi
tính chất.

## 4. Mọi phép toán trên roofline của máy

Biểu đồ roofline vẽ mái nhà $\min(R_\text{peak},\; I \cdot \text{BW})$ từ hai con số ở phần 1, rồi đặt
từng phép toán đã đo vào đúng mật độ tính toán của nó. Mật độ ở đây tính theo số byte tối thiểu, mỗi toán
hạng đi qua bộ nhớ đúng một lần, là một **giả định** của sổ tay.

In [ ]:
phep = {   # tên: (mật độ tính toán, phép tính mỗi giây đo được)
    "cộng hai mảng 128 MB": (1 / (3 * B4), N_ADD / t_add),
    "tầng tuyến tính, batch 1": (mat_do(1), flop_tt(1) / t_b[CAC_B.index(1)]),
    "tầng tuyến tính, batch 16": (mat_do(16), flop_tt(16) / t_b[CAC_B.index(16)]),
    "tầng tuyến tính, batch 256": (mat_do(256), flop_tt(256) / t_b[CAC_B.index(256)]),
    "nhân ma trận 1024 × 1024": (2 * N_MM ** 3 / (3 * B4 * N_MM ** 2), R_DINH),
}
assert abs(phep["nhân ma trận 1024 × 1024"][0] - N_MM / 6) < 1e-9   # I = N/6 cho ma trận vuông FP32
for ten, (I, r) in phep.items():
    do_tren_may(f"{ten:<28} mật độ {I:8.3f} FLOP/byte", r / 1e9, "GFLOP/s")

I = np.logspace(-2, 3, 300)
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.loglog(I, np.minimum(R_DINH, I * BW) / 1e9, color="0.25", lw=2, label="mái nhà từ hai con số đo được")
ax.axvline(DIEM_GAY, color="0.6", lw=0.9, ls=":")
ax.text(DIEM_GAY * 1.1, R_DINH / 1e9 * 0.02, "điểm gãy", fontsize=8, color="0.4")
mau_p = [MAU[1], MAU[1], MAU[3], MAU[0], MAU[0]]
lech = [(8, -14), (8, -14), (0, -18), None, (8, 8)]   # None: nhãn đặt xa điểm, có đường dẫn
for (ten, (Ii, r)), mau, d in zip(phep.items(), mau_p, lech):
    ax.plot([Ii], [r / 1e9], "o", color=mau, ms=9, mec="white", mew=1.5)
    if d is None:   # hai điểm xanh nằm sát nhau, nên nhãn này xuống hẳn phía dưới
        ax.annotate(ten, (Ii, r / 1e9), xytext=(Ii, r / 1e9 / 6), fontsize=8, ha="center",
                    arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
    else:
        ax.annotate(ten, (Ii, r / 1e9), xytext=d, textcoords="offset points", fontsize=8,
                    ha="right" if d[0] < 0 else ("center" if d[0] == 0 else "left"))
ax.set_xlim(1e-2, 1e3)
ax.set_ylim(R_DINH / 1e9 * 1e-3, R_DINH / 1e9 * 4)
ax.set_xlabel("mật độ tính toán (FLOP mỗi byte, thang log)")
ax.set_ylabel("GFLOP/s")
ax.set_title("Roofline của máy này (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Trên một máy có điểm gãy trên 0.5 FLOP mỗi byte, như hầu hết mọi máy, phép cộng và tầng tuyến tính ở
batch 1 nằm ở sườn dốc bên trái: kiểu Bandwidth Hog, nhanh lên chỉ khi chở ít byte hơn hoặc dùng lại byte
tốt hơn. Phép nhân ma trận vuông, khoảng 171 FLOP mỗi byte, nằm dưới phần mái phẳng trừ khi điểm gãy còn
cao hơn thế: kiểu Compute Beast, nhanh lên chỉ khi có thêm số học hoặc dùng số học tốt hơn. Một điểm càng nằm xa dưới mái thì
$\eta_\text{hw}$ của nó càng thấp.

## 5. Sparse Scatter: cùng số byte, khác chỗ lấy

Chương nói kiểu Sparse Scatter bị chi phối bởi những lần tra bảng không đều và tính cục bộ cache kém, như
các hệ gợi ý có bảng embedding (embedding table) khổng lồ. Ô dưới dựng một bảng 512 MB, mỗi hàng 16 số FP32
(64 byte), rồi lấy 65,536 hàng theo ba cách: các hàng liền nhau, các hàng ngẫu nhiên trong 1 MB đầu bảng
(đủ nhỏ để có thể nằm trong cache), và các hàng ngẫu nhiên trên cả 512 MB. Phép lấy hàng là đúng việc
`F.embedding` làm; sổ tay gọi `torch.index_select` với bộ đệm ra cấp sẵn, để không đo cả việc cấp phát. Cả
ba cách chở đúng một số byte và không làm phép tính nào.

**Dự đoán:** cách nào nhanh nhất, cách nào chậm nhất?

In [ ]:
del WT
D_EMB = 16
N_HANG = (512 << 20) // (B4 * D_EMB)
bang = torch.full((N_HANG, D_EMB), 1.0)        # 512 MB, lớn hơn mọi cache
M_TRA = 1 << 16
HANG_1MB = (1 << 20) // (B4 * D_EMB)            # số hàng trong 1 MB đầu bảng
dat_hat_giong()
thu_tu = {"liền nhau": torch.arange(M_TRA) + N_HANG // 3,
          "ngẫu nhiên trong 1 MB": torch.randint(0, HANG_1MB, (M_TRA,)),
          "ngẫu nhiên trên 512 MB": torch.randint(0, N_HANG, (M_TRA,))}
ra = torch.empty(M_TRA, D_EMB)
byte_tra = M_TRA * D_EMB * B4
bw_tra = {}
for ten, chi_so in thu_tu.items():
    t = thoi_gian(lambda: torch.index_select(bang, 0, chi_so, out=ra), lap=9)
    bw_tra[ten] = byte_tra / t
    do_tren_may(f"tra bảng, {ten}: mỗi hàng {t / M_TRA * 1e9:6.1f} ns, băng thông dùng được", byte_tra / t / 1e9, "GB/s")
assert torch.equal(ra, torch.full((M_TRA, D_EMB), 1.0))   # lấy đúng hàng, không phải đo một phép rỗng
del bang
do_tren_may("liền nhau so với ngẫu nhiên trên 512 MB", bw_tra["liền nhau"] / bw_tra["ngẫu nhiên trên 512 MB"], "lần")
do_tren_may("ngẫu nhiên trong 1 MB so với trên 512 MB",
            bw_tra["ngẫu nhiên trong 1 MB"] / bw_tra["ngẫu nhiên trên 512 MB"], "lần")

fig, ax = plt.subplots(figsize=(8, 3))
nhan = ["cộng hai mảng 128 MB (phần 1)"] + [f"tra bảng, {k}" for k in bw_tra]
gia = [BW] + list(bw_tra.values())
ax.barh(nhan, np.array(gia) / 1e9, color=["0.55", MAU[2], MAU[3], MAU[1]], height=0.6)
for i, v in enumerate(gia):
    ax.text(v / 1e9 * 1.02, i, f"{v / 1e9:.3g}", va="center", fontsize=8)
ax.invert_yaxis()
ax.set_xlim(0, max(gia) / 1e9 * 1.15)
ax.set_xlabel("băng thông dùng được (GB/s)")
ax.set_title("Cùng số byte, khác chỗ lấy (đo trên máy này)")
plt.tight_layout()
plt.show()

Ba lần tra bảng chở cùng một số byte và không làm phép tính nào, nên mọi khác biệt về thời gian giữa chúng
đến từ chỗ lấy. Các hàng liền nhau để bộ nạp trước của phần cứng đoán được hàng tiếp theo. Các hàng ngẫu
nhiên trong 1 MB có thể đã nằm sẵn trong cache. Các hàng ngẫu nhiên trên cả 512 MB thì mỗi hàng có thể là
một lần trượt cache, ở một chỗ phần cứng không đoán trước được. Trên roofline, phép này không có chỗ, vì
nó không làm phép tính nào: đây là lý do chương tách Sparse Scatter thành kiểu riêng, nơi dung lượng bộ
nhớ và độ trễ truy cập quyết định, chứ không phải FLOP. Một bảng embedding trên 100 GB, như của DLRM, lớn
hơn mọi cache hàng nghìn lần.

## 6. Một token GPT-2 trên máy này

Sinh văn bản tự hồi quy là ví dụ chương dùng cho Bandwidth Hog: mỗi token phải chở toàn bộ trọng số qua bộ
nhớ. Với 1.5 tỉ tham số, sổ tay **giả định** batch 1, mỗi tham số được đọc đúng một lần mỗi token và tốn
2 phép tính, bỏ qua bộ nhớ đệm của các token trước. Hai cận dưới của thời gian một token, theo hai con số đo
ở phần 1, là

$$T_\text{mem} = \frac{\text{số tham số} \times \text{byte mỗi tham số}}{\text{BW}},\qquad
T_\text{comp} = \frac{2 \times \text{số tham số}}{R_\text{peak}}.$$

Trước đó, kiểm lại mật độ tính toán của ResNet-50 trong Napkin Math 1.2, rồi đặt nó lên máy này.

In [ ]:
R50 = sach(8.2, nguon="8.2 GFLOP, 25.6 triệu tham số, 102.4 MB") * 1e9
R50_BYTE_FP16 = sach(5.63, trich=r"5.63 \times 10^{7}") * 1e7
theo_sach("ResNet-50, FP16: phép tính mỗi byte", R50 / R50_BYTE_FP16, sach=146, trich="146 FLOP/byte")
i_r50_fp32 = R50 / (2 * R50_BYTE_FP16)        # giả định: ở FP32 mọi byte gấp đôi
print(f"ở FP32 trên CPU: {i_r50_fp32:.0f} FLOP mỗi byte → "
      f"{'nghẽn ở tính toán' if i_r50_fp32 > DIEM_GAY else 'nghẽn ở bộ nhớ'} trên máy này "
      f"(so với điểm gãy đo được {DIEM_GAY:.3g})")

GPT2 = sach(1.5e9, nguon="1.5 tỉ / 7–70 tỉ")
print(f"\nGPT-2, {GPT2 / 1e9:.1f} tỉ tham số, batch 1:")
for ten, byte_ts in (("FP32", 4), ("INT8", 1)):
    t_mem, t_comp = GPT2 * byte_ts / BW, 2 * GPT2 / R_DINH
    do_tren_may(f"{ten}: chở trọng số {t_mem * 1e3:7.1f} ms, tính {t_comp * 1e3:7.1f} ms; nhanh nhất",
                1 / max(t_mem, t_comp), "token/s")
    print(f"      → {'nghẽn ở bộ nhớ' if t_mem > t_comp else 'nghẽn ở tính toán'}: số hạng lớn hơn quyết định "
          f"(đo trên máy này)")
print(f"(INT8 ở đây giả định tốc độ tính không đổi; trọng số FP32 của GPT-2 chiếm {GPT2 * 4 / 1e9:.0f} GB)")

Mật độ tính toán của một bước sinh token chỉ khoảng 0.5 phép tính mỗi byte ở FP32, thấp hơn điểm gãy của
hầu hết mọi bộ xử lý. Khi đó, mua thêm FLOP không giúp gì; chở ít byte hơn mới giúp, đúng điều chương nói:
giảm byte và hạ độ chính xác số (numerical precision) có thể thắng việc mua thêm thông lượng đỉnh. Khi đã chở ít byte hơn, điểm
nghẽn có thể chuyển sang số hạng kia: hãy xem dòng INT8 ở trên nói gì trên máy bạn. Cả hai dòng đều là cận
dưới của thời gian một token, vì chúng coi $\eta_\text{hw} = 1$ và chỉ đếm trọng số.

## Thử thêm

1. Ở phần 3, đổi `K_IN, M_OUT` thành `1024, 1024`, để trọng số chỉ còn 4 MB, đủ nhỏ để có thể nằm trong
   cache. Ở batch nhỏ, đường đo có còn bám đường dự đoán dựng từ băng thông của mảng 128 MB không?
2. Ở phần 5, đổi `D_EMB` thành 64, rồi 256. Khi mỗi hàng dài hơn một dòng cache, khoảng cách giữa tra liền
   nhau và tra ngẫu nhiên thay đổi thế nào?
3. Thêm `torch.set_num_threads(2)` (hoặc số nhân máy bạn có) ở đầu phần 1 và chạy lại phần 1. Con số nào
   tăng nhiều hơn, thông lượng đỉnh hay băng thông? Điểm gãy dịch về phía nào?

## Tóm lại

* Hai con số đo được, thông lượng đỉnh và băng thông, cho điểm gãy của chính máy bạn; phép toán có mật độ
  tính toán thấp hơn điểm gãy thì nghẽn ở bộ nhớ, và bộ xử lý nhanh hơn không giúp nó.
* Định luật sắt hiện ra ngay trên một phép cộng: mảng nhỏ bị phụ trội cố định chi phối, mảng lớn bị
  $D_\text{vol}/\text{BW}$ chi phối, và cache có thể tạo một đoạn nhanh hơn ở giữa.
* Tăng batch dùng lại trọng số, đẩy một tầng tuyến tính từ nghẽn ở bộ nhớ sang nghẽn ở tính toán; thời gian
  mỗi mẫu giảm, nhưng độ trễ cả batch tăng.
* Cùng số byte, thứ tự truy cập vẫn đổi được băng thông dùng được: đó là kiểu Sparse Scatter, nằm ngoài
  roofline.
* Một bước sinh token ở batch 1 có mật độ tính toán rất thấp, nên ước lượng tốc độ sinh token bắt đầu từ số
  byte trọng số chia cho băng thông.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Hệ thống học máy, bốn môi trường triển khai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch02-ml-systems/), dựng từ chương
[*ML Systems*](https://mlsysbook.ai/vol1/ml_systems/ml_systems.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.